temporary notes from doc : 
* dataset is small (132 instances) -> find right train/validation/test split and use cross-validation
* use cross entropy loss
* softmax before cross entropy for multiclass prediction
* word2vec where low-vector distance means same context , t-sne embedding for visualisation
* sigmoid activation used for gates in lstm
* tanh used for state transformations
* first recurrent input starts from a null vector
* lstm core idea : one cell reads one word at a time step,update hidden memory, the same weights are reused at every time step
* shared parameters are updated from all time steps
* LSTM gates to retain and propagate information, while in RNN the influence of an early input decreases over time
* LSTM : a cell state c (the memory),3 gates : forget gate(what to forget),input gate(what to write),output gate(what to reveal to the next cell)
  

In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import tensorflow as tf
import random

import numpy as np
import pandas as pd
import os
import urllib.request
import zipfile

try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

#fix random seed to make results more stable
#pour avoir presque les mêmes résultats à chaque execution
def reset_random_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

reset_random_seed(42)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# Load data
data_path="../data/"
df=pd.read_csv(os.path.join(data_path, "train_emoji_enrichi_aug.csv"))
df.head()


I0000 00:00:1789805651.490221  260835 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789805651.544147  260835 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1789805652.957066  260835 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


W0000 00:00:1789805656.006713  260969 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1789805656.330090  260835 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


,sentence,emoji
0,never talk to me again,3
1,I am proud of your achievements,2
2,It is the worst day in my life,3
3,Miss you so much,0
4,food is life,4


In [2]:
x = df['sentence'].values #sentences 
y = df['emoji'].values #emoji (labels)

#tokenisation - decouper le texte en mots encodés en int
tokenizer = Tokenizer()
tokenizer.fit_on_texts(x)
x_sequences = tokenizer.texts_to_sequences(x)

#construire le vocabulaire - tout les mots connus par le modele
vocab_size=len(tokenizer.word_index)+1 

#padding - rendre toutes les sequences de même longueur pour entrainement en batch
max_len=max(len(seq) for seq in x_sequences)
X=pad_sequences(x_sequences,maxlen=max_len,padding="pre")

#cross validation split
#5 folds car le dataset est petit donc un seul split n'est pas suffisant
#stratified pour garder presque la même repartition des classes dans chaque fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [3]:
#Pre-trained embeddings: GloVe 50d
#on utilise des vecteurs deja appris sur un grand corpus pour donner au modele une information semantique de depart
embedding_dim = 50
glove_filename = f"glove.6B.{embedding_dim}d.txt"
glove_zip_path = os.path.join(data_path, "glove.6B.zip")
glove_path = os.path.join(data_path, glove_filename)
glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
embedding_cache_path = os.path.join(data_path, f"embedding_matrix_glove_{embedding_dim}d.npy")

if os.path.exists(embedding_cache_path):
    embedding_matrix = np.load(embedding_cache_path)
    matched_words = int(np.count_nonzero(np.linalg.norm(embedding_matrix, axis=1)))
else:
    if not os.path.exists(glove_path):
        if not os.path.exists(glove_zip_path):
            print("Downloading GloVe this takes few minutes it's a large file.")
            urllib.request.urlretrieve(glove_url, glove_zip_path)
        print("Extracting only the 50d GloVe file...")
        with zipfile.ZipFile(glove_zip_path, "r") as zip_ref:
            zip_ref.extract(glove_filename, data_path)

    #matrice d'embedding alignee avec les indices du tokenizer
    #on ne garde que les mots utiles au dataset au lieu de charger tout GloVe en memoire
    needed_words = set(tokenizer.word_index.keys())
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    matched_words = 0

    with open(glove_path, encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            if word in needed_words:
                index = tokenizer.word_index[word]
                embedding_matrix[index] = np.asarray(values[1:], dtype="float32")
                matched_words += 1
                if matched_words == len(needed_words):
                    break

    np.save(embedding_cache_path, embedding_matrix)

coverage = matched_words / len(tokenizer.word_index)
print(f"GloVe vocabulary coverage: {matched_words}/{len(tokenizer.word_index)} ({coverage:.2%})")


Extracting only the 50d GloVe file...


GloVe vocabulary coverage: 360/374 (96.26%)


## Évaluation 1 : validation croisée à 5 folds

Les trois modèles sont d'abord comparés par validation croisée stratifiée à 5 folds sur `train_emoji_enrichi_aug.csv`.

In [4]:
#Baseline: logistic regression
#baseline simple sans deep learning pour comparer avec RNN et LSTM
log_reg_model = make_pipeline(
    #bag of words - compter les mots dans chaque phrase
    CountVectorizer(),
    #modele simple de classification multiclass
    LogisticRegression(max_iter=1000)
)

log_reg_scores = cross_val_score(log_reg_model, x, y, cv=cv, scoring='accuracy')

print(f"Logistic regression CV scores: {log_reg_scores}")
print(f"Logistic regression mean accuracy: {log_reg_scores.mean():.4f}")


Logistic regression CV scores: [0.65454545 0.67272727 0.78181818 0.78181818 0.77777778]
Logistic regression mean accuracy: 0.7337


In [5]:
#Baseline: simple RNN with cross validation
rnn_scores = []

for fold, (train_index, val_index) in enumerate(cv.split(X, y), start=1):
    #reset le modele et les seeds à chaque fold
    tf.keras.backend.clear_session()
    reset_random_seed(42)

    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    #definition du modele - RNN simple
    rnn_model = Sequential([
        #embedding -- transformer chaque mot en vecteur
        Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False, mask_zero=True),
        #simpleRNN pour avoir une baseline plus simple que LSTM
        SimpleRNN(32),
        #5 classes donc 5 sorties avec softmax
        Dense(5, activation='softmax')
    ])

    rnn_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    rnn_model.fit(
        X_train, y_train,
        #30 epochs comme choix simple pour laisser le modele apprendre
        epochs=30,
        batch_size=16,
        #pas de shuffle pour garder les résultats plus stables
        shuffle=False,
        #verbose=0 pour ne pas afficher toutes les epochs
        verbose=0
    )

    rnn_loss, rnn_accuracy = rnn_model.evaluate(X_val, y_val, verbose=0)
    rnn_scores.append(rnn_accuracy)
    print(f"RNN fold {fold} accuracy: {rnn_accuracy:.4f}")

rnn_scores = np.array(rnn_scores)
print(f"RNN mean accuracy: {rnn_scores.mean():.4f}")


E0000 00:00:1789806804.201423  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


E0000 00:00:1789806805.345206  260835 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


RNN fold 1 accuracy: 0.5636


E0000 00:00:1789806812.707718  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


RNN fold 2 accuracy: 0.6000


RNN fold 3 accuracy: 0.6182


E0000 00:00:1789806817.906230  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


RNN fold 4 accuracy: 0.6909


E0000 00:00:1789806826.402051  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


RNN fold 5 accuracy: 0.7593
RNN mean accuracy: 0.6464


In [6]:
# LSTM with cross validation
lstm_scores = []

for fold, (train_index, val_index) in enumerate(cv.split(X, y), start=1):
    #reset le modele et les seeds à chaque fold
    tf.keras.backend.clear_session()
    reset_random_seed(42)

    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    #definition du modele - LSTM
    model = Sequential([
        #embedding -- transformer chaque mot en vecteur 
        Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False, mask_zero=True),
        #LSTM pour mieux garder l'information dans la phrase
        LSTM(32),
        #5 classes donc 5 sorties avec softmax
        Dense(5, activation='softmax')
    ])

    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(
        X_train, y_train,
        #30 epochs comme choix simple pour laisser le modele apprendre
        epochs=30,
        batch_size=16,
        #pas de shuffle pour garder les résultats plus stables
        shuffle=False,
        verbose=0
    )

    loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
    lstm_scores.append(accuracy)
    print(f"LSTM fold {fold} accuracy: {accuracy:.4f}")

lstm_scores = np.array(lstm_scores)
print(f"LSTM mean accuracy: {lstm_scores.mean():.4f}")


E0000 00:00:1789806832.145083  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM fold 1 accuracy: 0.6909


E0000 00:00:1789806837.961647  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM fold 2 accuracy: 0.6182


E0000 00:00:1789806843.471539  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM fold 3 accuracy: 0.6909


E0000 00:00:1789806849.304359  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM fold 4 accuracy: 0.6727


E0000 00:00:1789806855.117124  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM fold 5 accuracy: 0.8148
LSTM mean accuracy: 0.6975


In [7]:
#comparaison side to side
print(f"Logistic Regression mean accuracy: {log_reg_scores.mean():.4f}")
print(f"RNN mean accuracy: {rnn_scores.mean():.4f}")
print(f"LSTM mean accuracy: {lstm_scores.mean():.4f}")

Logistic Regression mean accuracy: 0.7337
RNN mean accuracy: 0.6464
LSTM mean accuracy: 0.6975


## Évaluation 2 : séparation train / test avec test_emoji.csv

En complément de la validation croisée ci-dessus (Évaluation 1), les trois modèles sont maintenant entraînés sur l'intégralité de `train_emoji_enrichi_aug.csv` puis évalués une seule fois sur `test_emoji.csv`, un jeu de phrases tenu à l'écart de l'entraînement et de l'augmentation.

In [8]:
df_test = pd.read_csv(os.path.join(data_path, "test_emoji.csv"), header=None, names=["sentence", "emoji"])
df_test["sentence"] = df_test["sentence"].str.strip()

x_test = df_test["sentence"].values
y_test = df_test["emoji"].values

x_test_sequences = tokenizer.texts_to_sequences(x_test)
X_test = pad_sequences(x_test_sequences, maxlen=max_len, padding="pre")

print(f"Test samples: {len(df_test)}")
print(df_test["emoji"].value_counts().sort_index())

Test samples: 56
emoji
0    12
1     5
2    18
3    15
4     6
Name: count, dtype: int64


In [9]:
#regression logistique entrainee sur tout le train, evaluee sur le test independant
log_reg_model_full = make_pipeline(
    CountVectorizer(),
    LogisticRegression(max_iter=1000)
)
log_reg_model_full.fit(x, y)

log_reg_test_accuracy = log_reg_model_full.score(x_test, y_test)
print(f"Logistic regression test accuracy: {log_reg_test_accuracy:.4f}")

Logistic regression test accuracy: 0.5893


In [10]:
#RNN entraine sur tout le train, evalue sur le test independant
tf.keras.backend.clear_session()
reset_random_seed(42)

rnn_model_full = Sequential([
    Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False, mask_zero=True),
    SimpleRNN(32),
    Dense(5, activation='softmax')
])

rnn_model_full.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

rnn_model_full.fit(
    X, y,
    epochs=30,
    batch_size=16,
    shuffle=False,
    verbose=0
)

rnn_test_loss, rnn_test_accuracy = rnn_model_full.evaluate(X_test, y_test, verbose=0)
print(f"RNN test accuracy: {rnn_test_accuracy:.4f}")

E0000 00:00:1789806860.768292  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


RNN test accuracy: 0.5893


In [11]:
#LSTM entraine sur tout le train, evalue sur le test independant
tf.keras.backend.clear_session()
reset_random_seed(42)

lstm_model_full = Sequential([
    Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False, mask_zero=True),
    LSTM(32),
    Dense(5, activation='softmax')
])

lstm_model_full.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

lstm_model_full.fit(
    X, y,
    epochs=30,
    batch_size=16,
    shuffle=False,
    verbose=0
)

lstm_test_loss, lstm_test_accuracy = lstm_model_full.evaluate(X_test, y_test, verbose=0)
print(f"LSTM test accuracy: {lstm_test_accuracy:.4f}")

E0000 00:00:1789806866.812084  260835 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM test accuracy: 0.5893


In [12]:
#comparaison side to side - evaluation 2 (test independant)
print(f"Logistic Regression test accuracy: {log_reg_test_accuracy:.4f}")
print(f"RNN test accuracy: {rnn_test_accuracy:.4f}")
print(f"LSTM test accuracy: {lstm_test_accuracy:.4f}")

Logistic Regression test accuracy: 0.5893
RNN test accuracy: 0.5893
LSTM test accuracy: 0.5893


Setup : 

3 modèles comparés pour prédire l'emoji d'une phrase :

-Régression logistique : un modèle simple sans deep learning, utilisé comme baseline non-DL. Étant donné la petite taille du dataset, ce modèle permet de vérifier si une approche plus simple peut déjà bien généraliser.

-RNN simple : un modèle deep learning séquentiel utilisé comme baseline DL. L’idée est de voir si le fait de traiter les phrases comme des séquences de mots apporte réellement une amélioration significative des performances. Les mots sont initialisés avec des embeddings pré-entraînés GloVe, avec une couverture de 96.26% du vocabulaire du dataset.

-LSTM : une version plus avancée du RNN, testée pour mieux gérer la circulation de l’information au fil de la séquence (grâce à ses mécanismes de portes). Il utilise aussi les embeddings pré-entraînés GloVe pour partir de représentations de mots déjà apprises sur un grand corpus.

Résultats & Interpretation : 

Les résultats montrent que le RNN simple est le moins performant (accuracy moyenne d’environ 0.65). Le LSTM fait mieux (une accuracy moyenne d’environ 0.70). Cette amélioration est logique, car le LSTM possède des portes qui lui permettent de mieux décider quelles informations garder, oublier ou transmettre d’un mot au suivant. Même si les phrases du dataset sont assez courtes, ce mécanisme peut quand même l’aider à mieux prendre en compte l’ordre des mots et l’importance de certains termes. Les embeddings pré-entraînés permettent aussi aux modèles séquentiels de partir de représentations de mots plus riches, au lieu de devoir apprendre le sens des mots uniquement à partir du dataset.

Cependant, le meilleur résultat est obtenu par la régression logistique, avec une accuracy moyenne d’environ 0.73. Cela peut s’expliquer surtout par la taille encore limitée du dataset, même après enrichissement et augmentation (274 exemples). Les modèles de deep learning comme le RNN et le LSTM ont plus de paramètres à apprendre, donc ils ont généralement besoin de beaucoup plus de données pour bien généraliser alors qu'avec peu d’exemples, leurs performances peuvent devenir instables selon les folds de validation croisée, et ils risquent plus facilement l'overfit. Même avec des embeddings pré-entraînés, le LSTM se rapproche de la baseline mais ne la dépasse pas.


Ainsi, dans ce cas précis le modèle le plus simple est le plus performant, un modèle plus complexe n’est pas forcément meilleur quand le dataset reste limité. L’augmentation des données permet d’avoir plus d’exemples que le dataset initial, mais elle ne suffit pas encore à rendre les modèles séquentiels meilleurs que la baseline.
